# Système de Prévision des Livraisons Client — Phase 1 : Data Engineering

---

| | |
|---|---|
| **Auteur** | Ada Makchi |
| **Encadrant** | — |
| **Formation** | Master — Spécialité Supply Chain / IA |
| **Période couverte** | 2021 – 2025 |
| **Date du rapport** | Avril 2026 |

---

## Contexte et problématique

L'entreprise GE génère chaque année des milliers de commandes clients. La question centrale du mémoire est :

> **Comment une approche IA peut-elle prédire si une livraison sera en retard, afin d'optimiser la chaîne d'approvisionnement ?**

Ce notebook constitue la **Phase 1 — Data Engineering** du projet. Il transforme deux extractions brutes (commandes et livraisons) en un dataset structuré, prêt à alimenter les modèles de Machine Learning de la Phase 2.

## Sources de données

| Fichier | Lignes | Colonnes | Description |
|---|---|---|---|
| `Extraction commande client 2021-2025.csv` | 352 549 | 14 | Commandes passées par les clients |
| `Extraction livraison client 2021-2025.csv` | 353 325 | 13 | Livraisons effectives correspondantes |

## Pipeline de traitement

```
Fichiers CSV bruts
      │
      ▼
 Étape 1 ─── Chargement, encodage, renommage snake_case
      │
      ▼
 Étape 2 ─── Traitement des valeurs manquantes
      │
      ▼
 Étape 3 ─── Normalisation et validation des dates
      │
      ▼
 Étape 4 ─── Jointure Commandes × Livraisons (LEFT JOIN)
      │
      ▼
 Étape 5 ─── Feature Engineering temporel + variable cible
      │
      ▼
 Étape 6 ─── Encodage des variables catégorielles
      │
      ▼
 Étape 7 ─── Vérification finale + export dataset_ml_final.parquet
```

**Résultat final :** `dataset_ml_final.parquet` — 349 390 lignes × 24 colonnes (23 features + 1 cible binaire `en_retard`)

---
## 0. Configuration de l'environnement

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder

# Configuration de l'affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

# Répertoire racine du projet (fonctionne depuis notebooks/ ou depuis la racine)
BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data', 'processed')
os.makedirs(DATA_DIR, exist_ok=True)

print(f'Répertoire racine : {BASE_DIR}')
print(f'Répertoire données : {DATA_DIR}')

Répertoire racine : c:\Users\lenovo\Desktop\Extraction livraison client 2021-2025
Répertoire données : c:\Users\lenovo\Desktop\Extraction livraison client 2021-2025\data\processed


---
## Étape 1 — Chargement, Encodage et Renommage

### 1.1 Diagnostic préliminaire

Avant tout chargement, nous avons effectué une **inspection binaire** des fichiers CSV pour identifier :

- **Encodage :** Les octets `0xC3 0xA9` correspondent au caractère `é` en **UTF-8** (et non latin-1 comme supposé initialement). Le chargement doit utiliser `encoding='utf-8'`.
- **Séparateur :** Le séparateur effectif est la **virgule** `,` et non le point-virgule `;`.
- **Colonne segment :** Le nom exact est `Ségment stratégique` avec un accent sur le **S** — une particularité à gérer lors du renommage.

### 1.2 Chargement des fichiers

In [2]:
df_cmd = pd.read_csv(
    os.path.join(BASE_DIR, 'Extraction commande client 2021-2025.csv'),
    encoding='utf-8',
    sep=','
)
df_liv = pd.read_csv(
    os.path.join(BASE_DIR, 'Extraction livraison client 2021-2025.csv'),
    encoding='utf-8',
    sep=','
)

print(f'Commandes  : {df_cmd.shape[0]:,} lignes × {df_cmd.shape[1]} colonnes')
print(f'Livraisons : {df_liv.shape[0]:,} lignes × {df_liv.shape[1]} colonnes')
print()
print('Aperçu des colonnes brutes — Commandes :')
for c in df_cmd.columns:
    print(f'  {repr(c)}')

Commandes  : 352,549 lignes × 14 colonnes
Livraisons : 353,325 lignes × 13 colonnes

Aperçu des colonnes brutes — Commandes :
  '#'
  'Num commande'
  "Numéro d'article"
  'Quantité'
  "Date d'enregistrement"
  'Date de livraison demandée'
  'Code client/fournisseur'
  'Activité stratégique client'
  'Activité stratégique article'
  'Ségment stratégique'
  'Type activité'
  'Pays/région'
  'Prix de vente'
  'Devise du prix'


### 1.3 Renommage en snake_case

Les colonnes brutes comportent des caractères accentués, des espaces et des apostrophes — incompatibles avec les conventions de nommage Python. Nous appliquons un renommage systématique en **snake_case** pour les deux datasets.

In [3]:
rename_cmd = {
    '#':                                'row_id',
    'Num commande':                     'num_commande',
    "Numéro d'article":                 'code_article',
    'Quantité':                         'qte_demandee',
    "Date d'enregistrement":            'date_enregistrement',
    'Date de livraison demandée':       'date_livraison_demandee',
    'Code client/fournisseur':          'code_client',
    'Activité stratégique client':      'famille_activite_client',
    'Activité stratégique article':     'famille_activite_article',
    'Ségment stratégique':              'segment',   # accent sur S (bug source)
    'Type activité':                    'type_activite',
    'Pays/région':                      'pays',
    'Prix de vente':                    'prix',
    'Devise du prix':                   'devise',
}
rename_liv = {
    '#':                                'row_id',
    'Num commande':                     'num_commande',
    "Numéro d'article":                 'code_article',
    'Qté':                              'qte_livree',
    'Date de livraison réelle':         'date_livraison_reelle',
    'Code client/fournisseur':          'code_client',
    'Activité stratégique client':      'famille_activite_client',
    'Activité stratégique article':     'famille_activite_article',
    'Ségment stratégique':              'segment',
    'Type activité':                    'type_activite',
    'Pays/région':                      'pays',
    'Prix de vente':                    'prix',
    'Devise du prix':                   'devise',
}

df_cmd = df_cmd.rename(columns=rename_cmd)
df_liv = df_liv.rename(columns=rename_liv)

print('Colonnes CMD :', list(df_cmd.columns))
print()
print('Colonnes LIV :', list(df_liv.columns))

Colonnes CMD : ['row_id', 'num_commande', 'code_article', 'qte_demandee', 'date_enregistrement', 'date_livraison_demandee', 'code_client', 'famille_activite_client', 'famille_activite_article', 'segment', 'type_activite', 'pays', 'prix', 'devise']

Colonnes LIV : ['row_id', 'num_commande', 'code_article', 'qte_livree', 'date_livraison_reelle', 'code_client', 'famille_activite_client', 'famille_activite_article', 'segment', 'type_activite', 'pays', 'prix', 'devise']


### 1.4 Vérification initiale — Valeurs nulles et structure

In [4]:
print('=== COMMANDES — Valeurs nulles ===')
print(df_cmd.isnull().sum().to_string())
print()
print('=== LIVRAISONS — Valeurs nulles ===')
print(df_liv.isnull().sum().to_string())

=== COMMANDES — Valeurs nulles ===
row_id                         0
num_commande                   0
code_article                  31
qte_demandee                   0
date_enregistrement            0
date_livraison_demandee        0
code_client                    0
famille_activite_client       17
famille_activite_article       0
segment                       19
type_activite                  0
pays                          99
prix                           0
devise                      1987

=== LIVRAISONS — Valeurs nulles ===
row_id                         0
num_commande                   0
code_article                  30
qte_livree                     0
date_livraison_reelle          0
code_client                    0
famille_activite_client       17
famille_activite_article       0
segment                       19
type_activite                  0
pays                         100
prix                           0
devise                      1977


In [5]:
df_cmd.to_csv(os.path.join(DATA_DIR, 'cmd_step1.csv'), index=False, encoding='utf-8')
df_liv.to_csv(os.path.join(DATA_DIR, 'liv_step1.csv'), index=False, encoding='utf-8')
print('Sauvegarde step1 : cmd_step1.csv, liv_step1.csv')

Sauvegarde step1 : cmd_step1.csv, liv_step1.csv


---
## Étape 2 — Traitement des Valeurs Manquantes

| Colonne | Nulls CMD | Nulls LIV | Stratégie retenue | Justification |
|---|---|---|---|---|
| `code_article` | 31 | 30 | **Suppression** | Article inconnu = ligne inutilisable pour le modèle |
| `segment` | 19 | 19 | **Résolu** par la suppression ci-dessus | 100% overlap avec `code_article` null |
| `pays` | 99 | 100 | **Mode par `code_client`**, sinon `UNKNOWN` | 98% des clients ont un pays connu par ailleurs |
| `devise` | 1 987 | 1 977 | **Mode par `pays`**, sinon `EUR` | Forte corrélation pays → devise (FR=EUR, US=USD…) |
| `famille_activite_client` | 17 | 17 | **Mode par `code_client`**, sinon `AUTRE` | Attribut stable par client |

### 2.1 — `code_article` : suppression des lignes null uniquement

Le modèle prédit au niveau `(code_client, code_article)`. Une ligne sans article est inutilisable. Les 31 (CMD) et 30 (LIV) lignes concernées représentent **0,009%** des données — la suppression est négligeable.

In [6]:
n_cmd, n_liv = len(df_cmd), len(df_liv)

df_cmd = df_cmd.dropna(subset=['code_article'])
df_liv = df_liv.dropna(subset=['code_article'])

print(f'CMD : {n_cmd:,} → {len(df_cmd):,}  ({n_cmd - len(df_cmd)} lignes supprimées)')
print(f'LIV : {n_liv:,} → {len(df_liv):,}  ({n_liv - len(df_liv)} lignes supprimées)')

CMD : 352,549 → 352,518  (31 lignes supprimées)
LIV : 353,325 → 353,295  (30 lignes supprimées)


### 2.2 — `pays` : imputation par mode du client

In [7]:
def impute_pays(df):
    mode_pays = (
        df.dropna(subset=['pays'])
          .groupby('code_client')['pays']
          .agg(lambda x: x.mode()[0] if len(x) > 0 else None)
    )
    n = df['pays'].isna().sum()
    df['pays'] = df.apply(
        lambda row: mode_pays.get(row['code_client'], 'UNKNOWN') if pd.isna(row['pays']) else row['pays'],
        axis=1
    )
    df['pays'] = df['pays'].fillna('UNKNOWN')
    print(f'  pays null : {n} → {df["pays"].isna().sum()} (mode client ou UNKNOWN)')
    return df

df_cmd = impute_pays(df_cmd)
df_liv = impute_pays(df_liv)

  pays null : 99 → 0 (mode client ou UNKNOWN)
  pays null : 100 → 0 (mode client ou UNKNOWN)


### 2.3 — `devise` : imputation par devise majoritaire du pays

In [8]:
def impute_devise(df):
    devise_maj = (
        df.dropna(subset=['devise'])
          .groupby('pays')['devise']
          .agg(lambda x: x.mode()[0] if len(x) > 0 else 'EUR')
    )
    n = df['devise'].isna().sum()
    df['devise'] = df.apply(
        lambda row: devise_maj.get(row['pays'], 'EUR') if pd.isna(row['devise']) else row['devise'],
        axis=1
    )
    df['devise'] = df['devise'].fillna('EUR')
    print(f'  devise null : {n} → {df["devise"].isna().sum()} (mode pays ou EUR)')
    return df

df_cmd = impute_devise(df_cmd)
df_liv = impute_devise(df_liv)

  devise null : 1981 → 0 (mode pays ou EUR)
  devise null : 1972 → 0 (mode pays ou EUR)


### 2.4 — `famille_activite_client` : imputation par mode du client

In [9]:
def impute_famille(df):
    mode_fam = (
        df.dropna(subset=['famille_activite_client'])
          .groupby('code_client')['famille_activite_client']
          .agg(lambda x: x.mode()[0] if len(x) > 0 else None)
    )
    n = df['famille_activite_client'].isna().sum()
    df['famille_activite_client'] = df.apply(
        lambda row: mode_fam.get(row['code_client'], 'AUTRE') if pd.isna(row['famille_activite_client']) else row['famille_activite_client'],
        axis=1
    )
    df['famille_activite_client'] = df['famille_activite_client'].fillna('AUTRE')
    print(f'  famille_activite_client null : {n} → {df["famille_activite_client"].isna().sum()}')
    return df

df_cmd = impute_famille(df_cmd)
df_liv = impute_famille(df_liv)

  famille_activite_client null : 17 → 0
  famille_activite_client null : 17 → 0


### 2.5 — Vérification finale Étape 2

In [10]:
nulls_cmd = df_cmd.isnull().sum()[df_cmd.isnull().sum() > 0]
nulls_liv = df_liv.isnull().sum()[df_liv.isnull().sum() > 0]

print('=== RÉSULTAT ÉTAPE 2 ===')
print(f'CMD shape : {df_cmd.shape}')
print(f'LIV shape : {df_liv.shape}')
print()
print('Nulls CMD restants :', nulls_cmd.to_dict() if len(nulls_cmd) > 0 else 'AUCUN')
print('Nulls LIV restants :', nulls_liv.to_dict() if len(nulls_liv) > 0 else 'AUCUN')

df_cmd.to_csv(os.path.join(DATA_DIR, 'cmd_step2.csv'), index=False, encoding='utf-8')
df_liv.to_csv(os.path.join(DATA_DIR, 'liv_step2.csv'), index=False, encoding='utf-8')
print('\nSauvegarde step2 : cmd_step2.csv, liv_step2.csv')

=== RÉSULTAT ÉTAPE 2 ===
CMD shape : (352518, 14)
LIV shape : (353295, 13)

Nulls CMD restants : AUCUN
Nulls LIV restants : AUCUN

Sauvegarde step2 : cmd_step2.csv, liv_step2.csv


---
## Étape 3 — Normalisation et Validation des Dates

Les trois colonnes de dates sont au format chaîne `dd/mm/yyyy`. Nous les convertissons en objets `datetime64` et vérifions la cohérence temporelle.

| Colonne | Source | Format brut |
|---|---|---|
| `date_enregistrement` | CMD | `dd/mm/yyyy` |
| `date_livraison_demandee` | CMD | `dd/mm/yyyy` |
| `date_livraison_reelle` | LIV | `dd/mm/yyyy` |

In [11]:
# Rechargement depuis step2 pour garantir la reproductibilité
df_cmd = pd.read_csv(os.path.join(DATA_DIR, 'cmd_step2.csv'), encoding='utf-8')
df_liv = pd.read_csv(os.path.join(DATA_DIR, 'liv_step2.csv'), encoding='utf-8')

In [12]:
df_cmd['date_enregistrement']     = pd.to_datetime(df_cmd['date_enregistrement'],     format='%d/%m/%Y', errors='coerce')
df_cmd['date_livraison_demandee'] = pd.to_datetime(df_cmd['date_livraison_demandee'], format='%d/%m/%Y', errors='coerce')
df_liv['date_livraison_reelle']   = pd.to_datetime(df_liv['date_livraison_reelle'],   format='%d/%m/%Y', errors='coerce')

print('NaT après conversion (0 attendu) :')
print(f'  date_enregistrement     : {df_cmd["date_enregistrement"].isna().sum()}')
print(f'  date_livraison_demandee : {df_cmd["date_livraison_demandee"].isna().sum()}')
print(f'  date_livraison_reelle   : {df_liv["date_livraison_reelle"].isna().sum()}')
print()
print(f'Plage CMD : {df_cmd["date_enregistrement"].min().date()} → {df_cmd["date_enregistrement"].max().date()}')
print(f'Plage LIV : {df_liv["date_livraison_reelle"].min().date()} → {df_liv["date_livraison_reelle"].max().date()}')

NaT après conversion (0 attendu) :
  date_enregistrement     : 0
  date_livraison_demandee : 0
  date_livraison_reelle   : 0

Plage CMD : 2020-09-23 → 2025-12-23
Plage LIV : 2021-01-04 → 2025-12-23


### 3.2 — Distribution annuelle des données

In [13]:
print('=== Commandes par année ===')
print(df_cmd['date_enregistrement'].dt.year.value_counts().sort_index().to_string())
print()
print('=== Livraisons par année ===')
print(df_liv['date_livraison_reelle'].dt.year.value_counts().sort_index().to_string())

=== Commandes par année ===
date_enregistrement
2020     1718
2021    75683
2022    67569
2023    69129
2024    71637
2025    66782

=== Livraisons par année ===
date_livraison_reelle
2021    76161
2022    66548
2023    70078
2024    72004
2025    68504


### 3.3 — Détection et suppression des anomalies temporelles

Une anomalie critique est détectée : **3 lignes ont une `date_livraison_reelle` antérieure à la `date_enregistrement`**.

Il s'agit de la commande 152881, enregistrée en septembre 2021 mais avec une livraison datée de juillet 2021 — une erreur de saisie tardive dans le système ERP.

**Décision : suppression** — un délai `delai_jours` négatif (livraison avant commande) est physiquement impossible et polluerait l'apprentissage du modèle. Perte : 3 lignes sur 353 292 = **0,0008%**.

In [14]:
# Jointure de contrôle pour détecter les anomalies
check = df_cmd[['num_commande','code_article','date_enregistrement']].merge(
    df_liv[['num_commande','code_article','date_livraison_reelle']],
    on=['num_commande','code_article'], how='inner'
)
anomalies = check[check['date_livraison_reelle'] < check['date_enregistrement']]
print(f'Anomalies détectées (livraison avant enregistrement) : {len(anomalies)}')
print(anomalies.to_string())

# Suppression dans df_liv
n_avant = len(df_liv)
cles_anomalie = set(zip(anomalies['num_commande'], anomalies['code_article']))
mask = df_liv.apply(lambda r: (r['num_commande'], r['code_article']) in cles_anomalie, axis=1)
df_liv = df_liv[~mask]
print(f'\nLIV : {n_avant:,} → {len(df_liv):,} ({n_avant - len(df_liv)} lignes supprimées)')

Anomalies détectées (livraison avant enregistrement) : 3
       num_commande code_article date_enregistrement date_livraison_reelle
45170        152881        A0858          2021-09-01            2021-07-02
45171        152881        A0989          2021-09-01            2021-07-02
45173        152881        A0592          2021-09-01            2021-07-02

LIV : 353,295 → 353,292 (3 lignes supprimées)


In [15]:
df_cmd.to_csv(os.path.join(DATA_DIR, 'cmd_step3.csv'), index=False, encoding='utf-8')
df_liv.to_csv(os.path.join(DATA_DIR, 'liv_step3.csv'), index=False, encoding='utf-8')
print('Sauvegarde step3 : cmd_step3.csv, liv_step3.csv')
print(f'CMD : {df_cmd.shape}  |  LIV : {df_liv.shape}')

Sauvegarde step3 : cmd_step3.csv, liv_step3.csv
CMD : (352518, 14)  |  LIV : (353292, 13)


---
## Étape 4 — Jointure Commandes × Livraisons

### Problématique des doublons

Les deux tables contiennent des **paires `(num_commande, code_article)` non uniques** :
- **CMD** : 2 790 paires dupliquées — causées par des ré-ouvertures de commandes dans l'ERP. **Agrégation :** `sum(qte_demandee)`, `max(prix)`.
- **LIV** : 3 779 paires dupliquées — causées par des **livraisons fractionnées** (plusieurs livraisons partielles pour une même ligne). **Agrégation :** `sum(qte_livree)`, `max(date_livraison_reelle)`.

### Choix de jointure

Une jointure **LEFT** (CMD → LIV) est appliquée pour **conserver les commandes non encore livrées**. Ces 332 lignes seront exclues de l'entraînement mais restent utiles pour les prédictions en production.

In [16]:
df_cmd = pd.read_csv(os.path.join(DATA_DIR, 'cmd_step3.csv'), encoding='utf-8',
                     parse_dates=['date_enregistrement','date_livraison_demandee'])
df_liv = pd.read_csv(os.path.join(DATA_DIR, 'liv_step3.csv'), encoding='utf-8',
                     parse_dates=['date_livraison_reelle'])

In [17]:
# Colonnes de métadonnées stables pour groupby
cols_meta = ['num_commande','code_article','date_enregistrement','date_livraison_demandee',
             'code_client','famille_activite_client','famille_activite_article',
             'segment','type_activite','pays','devise']

df_cmd_agg = df_cmd.groupby(cols_meta, as_index=False).agg(
    qte_demandee=('qte_demandee', 'sum'),
    prix=('prix', 'max')
)
df_liv_agg = df_liv.groupby(['num_commande','code_article'], as_index=False).agg(
    qte_livree=('qte_livree', 'sum'),
    date_livraison_reelle=('date_livraison_reelle', 'max')
)

print(f'CMD après agrégation : {df_cmd_agg.shape}  (supprimé {df_cmd.shape[0] - df_cmd_agg.shape[0]} doublons)')
print(f'LIV après agrégation : {df_liv_agg.shape}  (supprimé {df_liv.shape[0] - df_liv_agg.shape[0]} doublons)')

CMD après agrégation : (349390, 13)  (supprimé 3128 doublons)
LIV après agrégation : (349091, 4)  (supprimé 4201 doublons)


In [18]:
# Jointure LEFT : toutes les commandes + livraisons quand disponibles
df_merged = pd.merge(
    df_cmd_agg,
    df_liv_agg,
    on=['num_commande', 'code_article'],
    how='left'
)
print(f'Shape après jointure : {df_merged.shape}')
print(f'Commandes livrées    : {df_merged["qte_livree"].notna().sum():,}')
print(f'Commandes en attente : {df_merged["qte_livree"].isna().sum():,}')

Shape après jointure : (349390, 15)
Commandes livrées    : 349,058
Commandes en attente : 332


### 4.3 — Calcul des métriques dérivées

Trois métriques métier sont calculées à partir de la jointure :

| Colonne | Formule | Rôle |
|---|---|---|
| `taux_satisfaction` | `qte_livree / qte_demandee` clippé en [0,1] | Taux de service |
| `livraison_excedentaire` | `qte_livree > qte_demandee` | Anomalie de sur-livraison |
| `delai_jours` | `date_livraison_reelle - date_livraison_demandee` | Retard (>0) ou avance (<0) en jours |

> **Note Data Leakage :** Ces colonnes seront **exclues des features ML** à l'Étape 7 — elles ne sont connues qu'après la livraison et ne peuvent pas être utilisées pour prédire.

In [19]:
# Statut de livraison
df_merged['statut'] = 'livre'
df_merged.loc[df_merged['qte_livree'].isna(), 'statut'] = 'en_attente'
df_merged['qte_livree'] = df_merged['qte_livree'].fillna(0)

# Métriques de qualité (post-livraison — exclues du ML)
df_merged['taux_satisfaction']    = (df_merged['qte_livree'] / df_merged['qte_demandee']).clip(0, 1)
df_merged['livraison_excedentaire'] = df_merged['qte_livree'] > df_merged['qte_demandee']
df_merged['delai_jours']          = (df_merged['date_livraison_reelle'] - df_merged['date_livraison_demandee']).dt.days

print('taux_satisfaction — min:', round(df_merged['taux_satisfaction'].min(), 3),
      ' max:', round(df_merged['taux_satisfaction'].max(), 3),
      ' mean:', round(df_merged['taux_satisfaction'].mean(), 3))
print('delai_jours       — min:', df_merged['delai_jours'].min(),
      ' max:', df_merged['delai_jours'].max(),
      ' median:', df_merged['delai_jours'].median())
print(f'Livraisons excédentaires : {df_merged["livraison_excedentaire"].sum()}')
print()
print(f'Shape final Étape 4 : {df_merged.shape}')

taux_satisfaction — min: 0.0  max: 1.0  mean: 0.999
delai_jours       — min: -124.0  max: 139.0  median: 0.0
Livraisons excédentaires : 55

Shape final Étape 4 : (349390, 19)


In [20]:
df_merged.to_csv(os.path.join(DATA_DIR, 'dataset_step4.csv'), index=False, encoding='utf-8')
df_merged.to_parquet(os.path.join(DATA_DIR, 'dataset_step4.parquet'), index=False)
print('Sauvegarde : dataset_step4.csv + dataset_step4.parquet')
print(f'Colonnes : {list(df_merged.columns)}')

Sauvegarde : dataset_step4.csv + dataset_step4.parquet
Colonnes : ['num_commande', 'code_article', 'date_enregistrement', 'date_livraison_demandee', 'code_client', 'famille_activite_client', 'famille_activite_article', 'segment', 'type_activite', 'pays', 'devise', 'qte_demandee', 'prix', 'qte_livree', 'date_livraison_reelle', 'statut', 'taux_satisfaction', 'livraison_excedentaire', 'delai_jours']


---
## Étape 5 — Feature Engineering Temporel

L'objectif est d'extraire des **variables prédictives** à partir des dates pour capturer la saisonnalité, les cycles budgétaires et le délai contractuel.

| Feature créée | Source | Description métier |
|---|---|---|
| `annee_cmd`, `mois_cmd`, `trimestre_cmd` | `date_enregistrement` | Saisonnalité de la commande |
| `semaine_cmd` | `date_enregistrement` | Semaine ISO (1–53) |
| `jour_semaine_cmd` | `date_enregistrement` | Jour (0=Lun … 6=Dim) |
| `est_fin_mois_cmd` | `date_enregistrement` | Pic de commandes en fin de mois (jour ≥ 25) |
| `annee_liv_dem`, `mois_liv_dem`, `trimestre_liv_dem` | `date_livraison_demandee` | Saisonnalité de la livraison souhaitée |
| `jour_semaine_liv_dem`, `est_weekend_liv_dem` | `date_livraison_demandee` | Livraison demandée un weekend |
| `delai_demande_jours` | `date_liv_dem - date_enreg` | Délai accordé par le client |
| **`en_retard`** | `delai_jours > 0` | **Variable cible** (0=à temps, 1=retard) |

In [21]:
df = pd.read_parquet(os.path.join(DATA_DIR, 'dataset_step4.parquet'))
print(f'Shape chargé : {df.shape}')

Shape chargé : (349390, 19)


In [22]:
# --- Features issues de date_enregistrement ---
df['annee_cmd']        = df['date_enregistrement'].dt.year
df['mois_cmd']         = df['date_enregistrement'].dt.month
df['trimestre_cmd']    = df['date_enregistrement'].dt.quarter
df['semaine_cmd']      = df['date_enregistrement'].dt.isocalendar().week.astype(int)
df['jour_semaine_cmd'] = df['date_enregistrement'].dt.dayofweek       # 0=Lundi, 6=Dimanche
df['est_fin_mois_cmd'] = (df['date_enregistrement'].dt.day >= 25).astype(int)

print('Distribution des commandes par mois :')
print(df['mois_cmd'].value_counts().sort_index().to_string())

Distribution des commandes par mois :
mois_cmd
1     37607
2     36388
3     33056
4     29976
5     26605
6     29102
7     26083
8     15944
9     29927
10    32325
11    28372
12    24005


In [23]:
# --- Features issues de date_livraison_demandee ---
df['annee_liv_dem']        = df['date_livraison_demandee'].dt.year
df['mois_liv_dem']         = df['date_livraison_demandee'].dt.month
df['trimestre_liv_dem']    = df['date_livraison_demandee'].dt.quarter
df['jour_semaine_liv_dem'] = df['date_livraison_demandee'].dt.dayofweek
df['est_weekend_liv_dem']  = (df['jour_semaine_liv_dem'] >= 5).astype(int)  # Sam=5, Dim=6

print(f'Livraisons demandées un weekend : {df["est_weekend_liv_dem"].sum():,} ({df["est_weekend_liv_dem"].mean()*100:.1f}%)')

Livraisons demandées un weekend : 504 (0.1%)


In [24]:
# --- Délai demandé par le client ---
df['delai_demande_jours'] = (df['date_livraison_demandee'] - df['date_enregistrement']).dt.days

# --- Variable cible : en_retard ---
# NaN pour les commandes en_attente (cible inconnue — exclues de l'entraînement)
df['en_retard'] = np.where(
    df['statut'] == 'en_attente',
    np.nan,
    (df['delai_jours'] > 0).astype(float)
)

print('=== VARIABLE CIBLE en_retard ===')
counts = df['en_retard'].value_counts(dropna=True)
total_livre = df['en_retard'].notna().sum()
print(f'  0 — À temps : {int(counts.get(0.0, 0)):>8,}  ({counts.get(0.0,0)/total_livre*100:.1f}%)')
print(f'  1 — Retard  : {int(counts.get(1.0, 0)):>8,}  ({counts.get(1.0,0)/total_livre*100:.1f}%)')
print(f'  NaN (en_attente) :  {df["en_retard"].isna().sum():>5,}')
print()
print(f'delai_demande_jours : min={df["delai_demande_jours"].min()}  max={df["delai_demande_jours"].max()}  median={df["delai_demande_jours"].median()}')

=== VARIABLE CIBLE en_retard ===
  0 — À temps :  281,975  (80.8%)
  1 — Retard  :   67,083  (19.2%)
  NaN (en_attente) :    332

delai_demande_jours : min=-61  max=388  median=5.0


In [25]:
df.to_csv(os.path.join(DATA_DIR, 'dataset_step5.csv'), index=False, encoding='utf-8')
df.to_parquet(os.path.join(DATA_DIR, 'dataset_step5.parquet'), index=False)
print(f'Sauvegarde step5 — Shape : {df.shape}')

Sauvegarde step5 — Shape : (349390, 32)


---
## Étape 6 — Encodage des Variables Catégorielles

Les algorithmes de Machine Learning ne traitent que des valeurs numériques. Deux stratégies sont appliquées selon la cardinalité des colonnes :

| Méthode | Colonnes | Cardinalité | Justification |
|---|---|---|---|
| **Binaire direct** | `statut` | 2 | `livre=1`, `en_attente=0` |
| **Label Encoding** | `devise`, `pays`, `famille_activite_client`, `famille_activite_article`, `segment`, `type_activite` | 2–79 | Faible cardinalité, entier arbitraire suffisant |
| **Frequency Encoding** | `code_client`, `code_article` | 3 465 / 1 068 | Haute cardinalité — One-Hot créerait >4 000 colonnes |

In [26]:
df = pd.read_parquet(os.path.join(DATA_DIR, 'dataset_step5.parquet'))

cat_cols = ['statut', 'devise', 'pays', 'famille_activite_client',
            'famille_activite_article', 'segment', 'type_activite',
            'code_client', 'code_article']

print('Cardinalités des colonnes catégorielles :')
for col in cat_cols:
    print(f'  {col:<35} : {df[col].nunique():>5} valeurs uniques')

Cardinalités des colonnes catégorielles :
  statut                              :     2 valeurs uniques
  devise                              :     3 valeurs uniques
  pays                                :    79 valeurs uniques
  famille_activite_client             :     4 valeurs uniques
  famille_activite_article            :     3 valeurs uniques
  segment                             :     9 valeurs uniques
  type_activite                       :    12 valeurs uniques
  code_client                         :  3465 valeurs uniques
  code_article                        :  1068 valeurs uniques


In [27]:
# Encodage binaire — statut
df['statut_enc'] = (df['statut'] == 'livre').astype(int)
print('statut_enc :', df.groupby('statut')['statut_enc'].first().to_dict())

statut_enc : {'en_attente': 0, 'livre': 1}


In [28]:
# Label Encoding — colonnes à faible cardinalité
label_cols = ['devise', 'pays', 'famille_activite_client',
              'famille_activite_article', 'segment', 'type_activite']

label_encoders = {}
for col in label_cols:
    le = LabelEncoder()
    df[f'{col}_enc'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    ex = dict(zip(le.classes_[:3], le.transform(le.classes_[:3]).tolist()))
    print(f'{col:<35} : {df[col].nunique()} classes → {col}_enc  (ex : {ex})')

devise                              : 3 classes → devise_enc  (ex : {'CNY': 0, 'EUR': 1, 'USD': 2})
pays                                : 79 classes → pays_enc  (ex : {'AD': 0, 'AE': 1, 'AR': 2})
famille_activite_client             : 4 classes → famille_activite_client_enc  (ex : {'AUTRE': 0, 'CONSTRUCTION': 1, 'ELEVAGE': 2})
famille_activite_article            : 3 classes → famille_activite_article_enc  (ex : {'CONSTRUCTION': 0, 'ELEVAGE': 1, 'RETRACTION': 2})
segment                             : 9 classes → segment_enc  (ex : {'BITUME': 0, 'CONNECTIQUE': 1, 'CONTENTION': 2})
type_activite                       : 12 classes → type_activite_enc  (ex : {'ACCESSOIRE': 0, 'CONSOMMABLE': 1, 'CONSOMMABLE BRASURE': 2})


In [29]:
# Frequency Encoding — code_client et code_article
# Principe : remplacer chaque valeur par son nombre d'occurrences dans le dataset.
# Avantage : capture l'importance relative d'un client/article sans explosion dimensionnelle.
for col in ['code_client', 'code_article']:
    freq = df[col].value_counts()
    df[f'{col}_freq'] = df[col].map(freq)
    print(f'{col}_freq — min={df[f"{col}_freq"].min()}  max={df[f"{col}_freq"].max()}  médiane={df[f"{col}_freq"].median()}')

print()
print('Top 5 clients par volume de commandes :')
print(df.groupby('code_client')['code_client_freq'].first().sort_values(ascending=False).head(5).to_string())

code_client_freq — min=1  max=4578  médiane=365.0
code_article_freq — min=1  max=11553  médiane=2573.0

Top 5 clients par volume de commandes :
code_client
C12289    4578
C10917    3964
C05933    3909
C12408    3617
C03917    3540


In [30]:
enc_cols = ['statut_enc'] + [f'{c}_enc' for c in label_cols] + ['code_client_freq', 'code_article_freq']

print(f'Shape : {df.shape}')
print(f'Colonnes encodées ajoutées ({len(enc_cols)}) : {enc_cols}')
print()
nulls = df[enc_cols].isnull().sum()
print('Nulls dans les colonnes encodées :', 'Aucun' if not nulls[nulls > 0].any() else nulls[nulls > 0].to_string())

df.to_parquet(os.path.join(DATA_DIR, 'dataset_step6.parquet'), index=False)
df.to_csv(os.path.join(DATA_DIR, 'dataset_step6.csv'), index=False, encoding='utf-8')
print('\nSauvegarde step6 : dataset_step6.parquet + dataset_step6.csv')

Shape : (349390, 41)
Colonnes encodées ajoutées (9) : ['statut_enc', 'devise_enc', 'pays_enc', 'famille_activite_client_enc', 'famille_activite_article_enc', 'segment_enc', 'type_activite_enc', 'code_client_freq', 'code_article_freq']

Nulls dans les colonnes encodées : Aucun

Sauvegarde step6 : dataset_step6.parquet + dataset_step6.csv


---
## Étape 7 — Vérification Finale et Export du Dataset ML

### Sélection des features

Seules les colonnes **utilisables au moment de la prédiction** sont conservées. Les colonnes suivantes sont exclues :

| Colonne exclue | Raison |
|---|---|
| `delai_jours` | **Data leakage** — connu seulement après livraison |
| `taux_satisfaction` | **Data leakage** — connu seulement après livraison |
| `livraison_excedentaire` | **Data leakage** — connu seulement après livraison |
| `date_enregistrement`, `date_livraison_demandee`, `date_livraison_reelle` | Déjà décomposées en features numériques |
| `code_client`, `code_article`, `num_commande` | Remplacés par frequency encoding ou sans pouvoir prédictif direct |
| `statut`, `devise`, `pays`, `famille_*`, `segment`, `type_activite` | Remplacées par colonnes `_enc` |
| `qte_livree` | Inconnue avant livraison |

In [31]:
df = pd.read_parquet(os.path.join(DATA_DIR, 'dataset_step6.parquet'))
print(f'Shape chargé : {df.shape}')

Shape chargé : (349390, 41)


In [32]:
FEATURE_COLS = [
    # Quantités et prix (connus à la commande)
    'qte_demandee', 'prix',
    # Features temporelles — date de commande
    'annee_cmd', 'mois_cmd', 'trimestre_cmd', 'semaine_cmd',
    'jour_semaine_cmd', 'est_fin_mois_cmd',
    # Features temporelles — date de livraison souhaitée
    'annee_liv_dem', 'mois_liv_dem', 'trimestre_liv_dem',
    'jour_semaine_liv_dem', 'est_weekend_liv_dem',
    # Délai accordé
    'delai_demande_jours',
    # Variables catégorielles encodées
    'statut_enc', 'devise_enc', 'pays_enc',
    'famille_activite_client_enc', 'famille_activite_article_enc',
    'segment_enc', 'type_activite_enc',
    # Frequency encoding
    'code_client_freq', 'code_article_freq',
]
TARGET_COL = 'en_retard'

df_ml = df[FEATURE_COLS + [TARGET_COL]].copy()

print(f'Dataset ML final : {df_ml.shape[0]:,} lignes × {df_ml.shape[1]} colonnes')
print(f'Features         : {len(FEATURE_COLS)}')
print(f'Cible            : {TARGET_COL}')

Dataset ML final : 349,390 lignes × 24 colonnes
Features         : 23
Cible            : en_retard


In [33]:
print('=== VÉRIFICATION FINALE ===')
print()

# Nulls dans les features
nulls_feat = df_ml[FEATURE_COLS].isnull().sum()
print('Nulls dans les features (X) :')
print(nulls_feat[nulls_feat > 0].to_string() if nulls_feat[nulls_feat > 0].any() else '  Aucun null')

print()

# Distribution de la cible
counts = df_ml[TARGET_COL].value_counts(dropna=True)
n_lab = df_ml[TARGET_COL].notna().sum()
print('=== DISTRIBUTION DE LA CIBLE ===')
print(f'  en_retard = 0  (à temps) : {int(counts.get(0.0, 0)):>8,}  ({counts.get(0.0,0)/n_lab*100:.1f}%)')
print(f'  en_retard = 1  (retard)  : {int(counts.get(1.0, 0)):>8,}  ({counts.get(1.0,0)/n_lab*100:.1f}%)')
print(f'  NaN (en attente)         : {df_ml[TARGET_COL].isna().sum():>8,}')
print()
print('  → Déséquilibre 80/20 — à traiter en Phase 2 (class_weight="balanced" ou SMOTE)')

=== VÉRIFICATION FINALE ===

Nulls dans les features (X) :
  Aucun null

=== DISTRIBUTION DE LA CIBLE ===
  en_retard = 0  (à temps) :  281,975  (80.8%)
  en_retard = 1  (retard)  :   67,083  (19.2%)
  NaN (en attente)         :      332

  → Déséquilibre 80/20 — à traiter en Phase 2 (class_weight="balanced" ou SMOTE)


In [34]:
print('=== TYPES DES FEATURES ===')
print(df_ml[FEATURE_COLS].dtypes.to_string())

=== TYPES DES FEATURES ===
qte_demandee                      int64
prix                            float64
annee_cmd                         int32
mois_cmd                          int32
trimestre_cmd                     int32
semaine_cmd                       int64
jour_semaine_cmd                  int32
est_fin_mois_cmd                  int64
annee_liv_dem                     int32
mois_liv_dem                      int32
trimestre_liv_dem                 int32
jour_semaine_liv_dem              int32
est_weekend_liv_dem               int64
delai_demande_jours               int64
statut_enc                        int64
devise_enc                        int64
pays_enc                          int64
famille_activite_client_enc       int64
famille_activite_article_enc      int64
segment_enc                       int64
type_activite_enc                 int64
code_client_freq                  int64
code_article_freq                 int64


In [35]:
# Export principal : Parquet (typage préservé, lecture rapide, ~10x plus léger que CSV)
df_ml.to_parquet(os.path.join(DATA_DIR, 'dataset_ml_final.parquet'), index=False)

# Export CSV de référence (lisibilité humaine)
df_ml.to_csv(os.path.join(DATA_DIR, 'dataset_ml_final.csv'), index=False, encoding='utf-8')

print('=== EXPORT TERMINÉ ===')
print(f'  dataset_ml_final.parquet')
print(f'  dataset_ml_final.csv')
print()
print('Fichiers produits dans data/processed :')
for f in sorted(os.listdir(DATA_DIR)):
    size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / (1024 * 1024)
    print(f'  {f:<45} {size_mb:>6.1f} MB')

=== EXPORT TERMINÉ ===
  dataset_ml_final.parquet
  dataset_ml_final.csv

Fichiers produits dans data/processed :
  cmd_step1.csv                                   38.2 MB
  cmd_step2.csv                                   38.2 MB
  cmd_step3.csv                                   38.2 MB
  dataset_ml_final.csv                            22.7 MB
  dataset_ml_final.parquet                         2.1 MB
  dataset_step4.csv                               47.5 MB
  dataset_step4.parquet                            3.3 MB
  dataset_step5.csv                               59.3 MB
  dataset_step5.parquet                            3.6 MB
  dataset_step6.csv                               67.2 MB
  dataset_step6.parquet                            4.6 MB
  liv_step1.csv                                   34.6 MB
  liv_step2.csv                                   34.6 MB
  liv_step3.csv                                   34.6 MB


---
## Bilan Phase 1 — Data Engineering

### Résumé du pipeline

| Étape | Action | Données entrantes | Données sortantes |
|---|---|---|---|
| 1 | Chargement + renommage | 352 549 CMD / 353 325 LIV | Colonnes normalisées |
| 2 | Valeurs manquantes | 0 null restants | 352 518 CMD / 353 295 LIV |
| 3 | Normalisation dates | Chaînes `dd/mm/yyyy` | `datetime64` — 3 anomalies supprimées |
| 4 | Jointure LEFT CMD × LIV | 2 tables | 349 390 lignes × 19 colonnes |
| 5 | Feature engineering | 19 colonnes | +13 features temporelles + cible `en_retard` |
| 6 | Encodage catégorielles | Chaînes brutes | +9 colonnes numériques |
| 7 | Sélection + export | 41 colonnes | **24 colonnes ML** |

### Dataset ML final

```
Fichier  : data/processed/dataset_ml_final.parquet
Shape    : 349 390 lignes × 24 colonnes
Features : 23 (utilisables à la date de commande, sans data leakage)
Cible    : en_retard (0 = à temps, 1 = retard)

Distribution cible :
  Classe 0 (à temps) : ~281 975  (80.8%)
  Classe 1 (retard)  :  ~67 083  (19.2%)
  NaN (en attente)   :      332  (exclus de l'entraînement)
```

### Décisions techniques clés

| Décision | Justification |
|---|---|
| Jointure LEFT (pas INNER) | Conserver les 332 commandes en attente pour prédictions futures |
| Suppression (et non imputation) des `code_article` null | Article inconnu = observation inutilisable métier |
| Frequency Encoding pour `code_client`/`code_article` | Cardinalité 3 465 / 1 068 — One-Hot créerait >4 000 colonnes parasites |
| Exclusion de `delai_jours`, `taux_satisfaction`, `livraison_excedentaire` | Data leakage — colonnes inconnues au moment de la prédiction |
| Variable cible `en_retard = (delai_jours > 0)` | Classification binaire plus robuste que régression sur `delai_jours` |

### Phase 2 — Modélisation (à venir)

1. Chargement de `dataset_ml_final.parquet`
2. Train/test split **stratifié** 80/20 (stratification sur `en_retard`)
3. Gestion du déséquilibre 80/19 : `class_weight='balanced'` ou SMOTE
4. Entraînement : **Random Forest**, **XGBoost**, **Régression Logistique**
5. Évaluation : F1-score, AUC-ROC, matrice de confusion
6. Interprétabilité : feature importance, SHAP values